# Habita MX · CrewAI + Ollama
Equipo 6 · Universidad Anáhuac Puebla

Ejecuta las celdas en orden dentro del ambiente del proyecto. Son cuatro agentes reales y datos simulados. Built with Llama. Los archivos oficiales del proyecto están en los módulos de `habita/`.

In [1]:
import sys, json, asyncio
from pathlib import Path
from IPython.display import display, Markdown
from habita.aplicacion import RAIZ, diagnosticar, ejecutar, reporte_markdown
print('Python:', sys.executable)
print('Proyecto:', RAIZ)

Python: C:\Users\krlan\OneDrive\Documentos\IA\Proyecto IA\habita-mx\.conda-env\python.exe
Proyecto: C:\Users\krlan\OneDrive\Documentos\IA\Proyecto IA\habita-mx


## 1. Comprobar el modelo local
Ollama debe estar abierto. El diagnóstico verifica el nombre del modelo y registra su digest.

In [2]:
diagnostico = diagnosticar()
diagnostico

{'base_url': 'http://127.0.0.1:11434',
 'modelo': 'llama3.2:latest',
 'digest': 'a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72',
 'detalles': {'parent_model': '',
  'format': 'gguf',
  'family': 'llama',
  'families': ['llama'],
  'parameter_size': '3.2B',
  'quantization_level': 'Q4_K_M',
  'context_length': 131072,
  'embedding_length': 3072,
  'runner': 'ggml'},
 'ollama': '0.40.14',
 'python': '3.11.18',
 'crewai': '1.15.35'}

## 2. Revisar y editar el perfil
Edita valores del diccionario antes de ejecutar. Los seis pesos deben sumar 1. Todos los montos son MXN. La descripción libre no reemplaza las cifras estructuradas.

In [3]:
perfil = json.loads((RAIZ / 'datos/perfil_demo.json').read_text(encoding='utf-8'))
perfil

{'descripcion': 'Busco una vivienda de al menos dos habitaciones y traslado de máximo 45 minutos. Me interesa equilibrar capacidad de pago, seguridad y exposición baja a riesgos.',
 'ingreso_mensual': 50000,
 'gastos_mensuales': 22000,
 'deudas_mensuales': 3000,
 'reserva_mensual': 3000,
 'presupuesto_maximo': 2000000,
 'enganche_disponible': 450000,
 'ahorro_gastos_iniciales': 120000,
 'pago_maximo_usuario': 14000,
 'habitaciones_minimas': 2,
 'traslado_maximo_min': 45,
 'seguridad_minima': 40,
 'riesgos_minimo': 40,
 'plazo_anios': 20,
 'tasa_anual': 0.105,
 'incremento_estres': 0.02,
 'limite_deuda_ingreso': 0.35,
 'enganche_minimo_pct': 0.2,
 'gastos_iniciales_pct': 0.05,
 'gasto_vivienda_mensual': 800,
 'cobertura_minima': 0.8,
 'pesos': {'finanzas': 0.4,
  'seguridad': 0.15,
  'riesgos': 0.15,
  'vivienda': 0.1,
  'movilidad': 0.1,
  'entorno': 0.1}}

In [4]:
# Ejemplo opcional: perfil['pago_maximo_usuario'] = 12000
# Se guarda una copia y se conserva el caso original.
perfil_notebook = RAIZ / 'datos/perfil_notebook.json'
perfil_notebook.write_text(json.dumps(perfil, ensure_ascii=False, indent=2), encoding='utf-8')
print(perfil_notebook)

C:\Users\krlan\OneDrive\Documentos\IA\Proyecto IA\habita-mx\datos\perfil_notebook.json


## 3. Ejecutar los cuatro agentes
Puede tardar desde decenas de segundos hasta varios minutos. Las líneas impresas muestran herramientas usadas y tareas aceptadas. Si falla, revisa `error.json`; no se sustituye por una simulación.

In [5]:
# Ejecutar en otro hilo mantiene compatible el ciclo de eventos de Jupyter.
resultado, carpeta = await asyncio.to_thread(ejecutar, perfil_path=perfil_notebook, modo='crew')
display(Markdown(reporte_markdown(resultado)))
print('Evidencia:', carpeta)

C:\Users\krlan\OneDrive\Documentos\IA\Proyecto IA\habita-mx\.conda-env\Lib\site-packages\pydantic\main.py:250: UserWarning: A function callback cannot be saved in a checkpoint; it is dropped and will not run after a restore. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


[herramienta] perfil


[tarea_completada] perfil


[validacion_rechazada] zonas


[herramienta] zonas


[tarea_completada] zonas


... (2 outputs left out)

[herramienta] coordinacion


[tarea_completada] coordinacion


# Habita MX - resultado

Modo: **crew**. Datos de demostración simulados.

Capacidad mensual: **$14,000.00 MXN**.

| Posición | Zona | Puntaje / 100 | Pago total MXN | Pago en estrés MXN |
|---|---|---:|---:|---:|
| 1 | Z02 - Zona Centro simulada | 52.38 | 13,778.94 | 15,569.83 |
| 2 | Z01 - Zona Norte simulada | 51.76 | 11,282.99 | 12,729.48 |

## Explicación del coordinador

La herramienta identificó Zona Z02 como la mejor opción, con un puntuación ponderada de 52.3816. El escenario de estrés supera el límite mensual en esta zona. Las otras opciones se descartan por restricciones, no por puntajes menores. La herramienta recomienda Zona Z02.

## Alertas y descartes

- Z02: El escenario de estrés supera el límite mensual.
- Z03: Precio superior al presupuesto máximo; Enganche menor al mínimo del escenario; Pago mensual superior a la capacidad calculada.
- Z04: Habitaciones insuficientes.
- Z05: Traslado superior al máximo.
- Z06: Cobertura de indicadores insuficiente; Falta indicador obligatorio: riesgos.

## Alcance

- Datos y condiciones financieras simulados para clase.
- La evaluación corresponde a la vivienda representativa de cada zona, no a toda su oferta.
- No es una oferta ni una aprobación de crédito.
- Las explicaciones del LLM requieren revisión humana.


Evidencia: C:\Users\krlan\OneDrive\Documentos\IA\Proyecto IA\habita-mx\salidas\ejecucion_20261105_135309_645966


## 4. Revisar la colaboración
El log contiene acciones observables y resúmenes finales, no cadenas internas de pensamiento.

In [6]:
eventos = [json.loads(l) for l in (carpeta / 'interacciones.jsonl').read_text(encoding='utf-8').splitlines()]
for evento in eventos:
    if evento['evento'] == 'tarea_completada':
        print(evento['etapa'], '<-', evento['contexto'])
        print(evento['conclusion']['resumen'], '\n')

perfil <- []
El límite mensual obligatorio es 14000.00 MXN; es un monto, no un porcentaje. Precio máximo, habitaciones mínimas y traslado máximo son restricciones obligatorias. Los pesos indican preferencias y no permiten violar restricciones. 

zonas <- ['perfil']
La herramienta evaluó el catálogo identificando las zonas que cumplen y las que no cumplen con los requisitos territoriales. Zona Z03 fue descartada por tener un precio superior al presupuesto máximo, mientras que Zona Z04 fue descartada por tener habitaciones insuficientes. Zona Z05 fue descartada por tener un traslado superior al máximo. La zona Z06 fue descartada por tener una cobertura de indicadores insuficiente y falta de un indicador obligatorio: riesgos. Las zonas Z01 y Z02 cumplen los requisitos territoriales. 

finanzas <- ['perfil']
El agente calcula el financiamiento de una vivienda con ingreso mensual de 50000 MXN, gastos mensuales de 22000 MXN, deudas mensuales de 3000 MXN y reserva mensual de 3000 MXN. El prec

## 5. Contrastar con el motor de cálculo
Esta línea base no ejecuta IA. Debe coincidir con los datos oficiales del Crew.

In [7]:
base, carpeta_base = ejecutar(perfil_path=perfil_notebook, modo='calculo')
assert resultado['resultado'] == base['resultado']
print('Ranking y cifras coinciden con la línea base.')

Ranking y cifras coinciden con la línea base.


## 6. Preguntas para la exposición
- ¿Por qué Z02 puede quedar arriba aunque Z01 tenga menor mensualidad?
- ¿Qué ocurre si el pago máximo baja a $10,000?
- ¿Por qué Z06 no puede recomendarse?
- ¿Qué hace el guardrail si el modelo inventa un ID?
- ¿Qué métricas de la Fase 2 aún no han sido evaluadas?

Consulta `docs/Fase_3.md` y `README.md` para especificación y referencias.